# Manual Feature Cleaning Review

This notebook reads `merged_feature_matrix.csv`, inspects one energy instrument step by step, and then applies the same cleaning rules to all four energy instruments: `cl1s`, `ho1s`, `rb1s`, and `ng1s`.

The final clean datasets are written to `data/features/clean_feature_set/`. These files are intended as the ready feature inputs for the later modelling/CPCV phase. The notebook does not standardise features, run CPCV, train models, or perform supervised feature selection.

## 1. Imports and paths

`ENERGY_INSTRUMENTS` controls the four datasets created at the end. `INSTRUMENT` is only the instrument shown in the review tables, so the notebook remains readable while the save step still loops through all energy tickers.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 160)

ENERGY_INSTRUMENTS = ["cl1s", "ho1s", "rb1s", "ng1s"]
INSTRUMENT = ENERGY_INSTRUMENTS[0]

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "data" / "features" / "merged_feature_matrix.csv").exists():
        ROOT = candidate
        break

FEATURE_MATRIX_PATH = ROOT / "data" / "features" / "merged_feature_matrix.csv"
FEATURE_CATALOG_PATH = ROOT / "data" / "features" / "feature_catalog.csv"
CLEAN_FEATURE_SET_DIR = ROOT / "data" / "features" / "clean_feature_set"
CLEAN_FEATURE_SET_PATHS = {
    instrument: CLEAN_FEATURE_SET_DIR / f"{instrument}_clean_feature_set.csv"
    for instrument in ENERGY_INSTRUMENTS
}
CLEAN_FEATURE_SET_PATH = CLEAN_FEATURE_SET_PATHS[INSTRUMENT]

HIGH_CORR_THRESHOLD = 0.95

# Low-variance removal threshold.
# Keep this at 0.0 to remove only constant numeric features.
# Use a tiny positive value such as 1e-8 for near-constant features.
# Avoid large raw-variance thresholds because features use different units/scales.
LOW_VARIANCE_THRESHOLD = 1e-8

print(f"Review instrument: {INSTRUMENT}")
print(f"Energy instruments: {ENERGY_INSTRUMENTS}")
print(f"Input:             {FEATURE_MATRIX_PATH}")
print(f"Feature catalog:   {FEATURE_CATALOG_PATH}")
print(f"Output folder:     {CLEAN_FEATURE_SET_DIR}")

**After this cell:** Paths are configured. The notebook will show detailed review tables for `INSTRUMENT`, then save one clean dataset per ticker in `ENERGY_INSTRUMENTS`.

## 2. Load data and filter to the review instrument

`merged_feature_matrix.csv` contains all instruments. We check the four energy instruments are present, show their coverage, and then filter to `INSTRUMENT` for the detailed review cells.

In [ ]:
full_matrix = pd.read_csv(FEATURE_MATRIX_PATH, parse_dates=["date"])
feature_catalog = pd.read_csv(FEATURE_CATALOG_PATH)
feature_group_map = feature_catalog.set_index("feature_name")["feature_group"].to_dict()

available_instruments = sorted(full_matrix["instrument"].unique())
missing_energy_instruments = sorted(set(ENERGY_INSTRUMENTS) - set(available_instruments))
assert not missing_energy_instruments, f"Missing energy instruments: {missing_energy_instruments}. Available: {available_instruments}"
assert INSTRUMENT in available_instruments, f"{INSTRUMENT} not found. Available: {available_instruments}"

energy_matrix = full_matrix.loc[full_matrix["instrument"].isin(ENERGY_INSTRUMENTS)].copy()
energy_coverage = (
    energy_matrix.groupby("instrument")
    .agg(
        rows=("date", "size"),
        columns=("instrument", lambda s: energy_matrix.shape[1]),
        min_date=("date", "min"),
        max_date=("date", "max"),
        active_signals=("primary_signal", lambda s: int(s.ne(0).sum())),
        duplicate_dates=("date", lambda s: int(s.duplicated().sum())),
    )
    .reindex(ENERGY_INSTRUMENTS)
)

raw = full_matrix.loc[full_matrix["instrument"].eq(INSTRUMENT)].reset_index(drop=True)
duplicate_dates = raw["date"].duplicated().sum()

print("Table: Energy instrument coverage")
display(energy_coverage)

print(f"Table: {INSTRUMENT} dataset summary")
summary = pd.DataFrame(
    {
        "value": [
            len(raw),
            raw.shape[1],
            raw["date"].min(),
            raw["date"].max(),
            duplicate_dates,
        ]
    },
    index=["rows", "columns", "min_date", "max_date", "duplicate_dates"],
)
display(summary)

print(f"Table: {INSTRUMENT} primary_signal distribution")
display(raw["primary_signal"].value_counts(dropna=False).sort_index().to_frame("count"))

print("Table: First rows")
display(raw.head())

**After this cell:**

Confirms the dataset for the selected instrument: row count, date range, signal distribution, and no duplicate dates.

## 3. Define ID, signal, and feature columns

`date` and `instrument` are identifiers. `primary_signal` is the trading signal â€” not a feature to clean. Everything else is a feature.

In [ ]:
id_cols = ["date", "instrument"]
signal_col = "primary_signal"
feature_cols = [col for col in raw.columns if col not in id_cols + [signal_col]]

numeric_feature_cols = raw[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
non_numeric_feature_cols = [col for col in feature_cols if col not in numeric_feature_cols]

feature_universe = pd.DataFrame(
    {
        "count": [len(id_cols), 1, len(feature_cols), len(numeric_feature_cols), len(non_numeric_feature_cols)]
    },
    index=["id_columns", "signal_columns", "feature_columns", "numeric_feature_columns", "non_numeric_feature_columns"],
)

print("Table: Feature universe summary")
display(feature_universe)

print("Table: Feature count by catalog group")
display(
    pd.Series(feature_cols, name="feature")
    .to_frame()
    .assign(feature_group=lambda x: x["feature"].map(feature_group_map).fillna("uncatalogued"))
    .groupby("feature_group")
    .size()
    .to_frame("feature_count")
    .sort_values("feature_count", ascending=False)
)

print("Non-numeric feature columns:")
print(non_numeric_feature_cols)

**After this cell:**

This defines the full feature universe. Non-numeric features stay in the review; they are not silently removed.

## 4. Missingness inspection

Before deciding what to drop, inspect which features are missing and how concentrated the missingness is.

In [ ]:
missing_pct = raw[feature_cols].isna().mean()
missing_count = raw[feature_cols].isna().sum()

missingness_table = (
    pd.DataFrame(
        {
            "feature": feature_cols,
            "feature_group": [feature_group_map.get(col, "uncatalogued") for col in feature_cols],
            "missing_count": [int(missing_count[col]) for col in feature_cols],
            "missing_pct": [missing_pct[col] for col in feature_cols],
        }
    )
    .sort_values(["missing_pct", "missing_count", "feature"], ascending=[False, False, True])
    .reset_index(drop=True)
)

print("Table: Features with missing values")
display(missingness_table.loc[missingness_table["missing_count"] > 0])

print("Table: Missingness by feature group")
display(
    missingness_table.groupby("feature_group")
    .agg(
        feature_count=("feature", "size"),
        features_with_missing=("missing_count", lambda s: int((s > 0).sum())),
        max_missing_pct=("missing_pct", "max"),
        mean_missing_pct=("missing_pct", "mean"),
    )
    .sort_values("max_missing_pct", ascending=False)
)

**After this cell:**

Missingness is tiny in percentage terms, but concentrated. The next section shows the exact rows and context instead of treating missingness as a generic feature-level problem.

## 5. Perfect Duplicate Groups

Exact duplicate features are the easiest first cleaning decision. Choose one feature to keep in each group and mark the rest to drop later. The notebook suggests HMM market-state probability names where those are available, because they are the most interpretable labels.

In [ ]:
normalised = raw[feature_cols].astype("string").fillna("<NA>")

parent = {feature: feature for feature in feature_cols}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    root_a = find(a)
    root_b = find(b)
    if root_a != root_b:
        parent[root_b] = root_a

seen_features = []
duplicate_feature_pairs = []
for col in feature_cols:
    duplicate_of = None
    for previous in seen_features:
        if normalised[col].equals(normalised[previous]):
            duplicate_of = previous
            union(previous, col)
            break
    if duplicate_of is not None:
        duplicate_feature_pairs.append({"feature": col, "duplicate_of": duplicate_of})
    else:
        seen_features.append(col)

duplicate_feature_pairs = pd.DataFrame(duplicate_feature_pairs, columns=["feature", "duplicate_of"])
duplicate_features = set(duplicate_feature_pairs["feature"]) if not duplicate_feature_pairs.empty else set()
duplicate_of_map = dict(zip(duplicate_feature_pairs["feature"], duplicate_feature_pairs["duplicate_of"])) if not duplicate_feature_pairs.empty else {}

clusters = {}
for feature in feature_cols:
    clusters.setdefault(find(feature), []).append(feature)
duplicate_clusters = [sorted(features) for features in clusters.values() if len(features) > 1]

preferred_duplicate_keep_order = [
    "hmm_prob_stress",
    "hmm_prob_calm_positive",
    "hmm_prob_calm_negative",
    "hmm_prob_upside_breakout",
    "hmm_regime_missing",
]

def suggest_duplicate_reason(features: list[str]) -> str:
    joined = " ".join(features)
    if "hmm_prob" in joined or "hmm_regime_prob" in joined:
        return "Exact duplicate HMM probability representation; prefer one interpretable market-state probability name."
    if "hmm_regime_missing" in joined or "_missing" in joined:
        return "Exact duplicate HMM missingness flag; keep one missing flag only."
    if any("hmm_regime_" in f for f in features) or any(f.startswith("hmm_market_") for f in features):
        return "Exact duplicate HMM one-hot/category representation; keep one representation only."
    return "Exact duplicate values across all rows; choose one feature to keep."

def suggested_keep_feature(features: list[str]) -> str:
    for preferred in preferred_duplicate_keep_order:
        if preferred in features:
            return preferred
    market_state_probs = [f for f in features if f.startswith("hmm_prob_") and f in {"hmm_prob_stress", "hmm_prob_calm_positive", "hmm_prob_calm_negative", "hmm_prob_upside_breakout"}]
    if market_state_probs:
        return sorted(market_state_probs)[0]
    return sorted(features)[0]

duplicate_group_rows = []
for group_id, features_in_group in enumerate(duplicate_clusters, start=1):
    suggested_keep = suggested_keep_feature(features_in_group)
    reason = suggest_duplicate_reason(features_in_group)
    for feature in features_in_group:
        duplicate_group_rows.append(
            {
                "duplicate_group_id": group_id,
                "feature": feature,
                "feature_group": feature_group_map.get(feature, "uncatalogued"),
                "missing_pct": missing_pct[feature],
                "unique_values": int(raw[feature].nunique(dropna=True)),
                "suggested_keep": suggested_keep,
                "suggested_reason": reason,
                "manual_keep": "",
                "manual_drop_reason": "",
            }
        )

duplicate_groups_df = pd.DataFrame(duplicate_group_rows)

print(f"Exact duplicate pairs: {len(duplicate_feature_pairs)}")
print(f"Exact duplicate groups: {len(duplicate_clusters)}")
print("Table: Exact duplicate feature pairs")
display(duplicate_feature_pairs)

for group_id in sorted(duplicate_groups_df["duplicate_group_id"].unique()):
    group_table = duplicate_groups_df.loc[duplicate_groups_df["duplicate_group_id"].eq(group_id)].copy()
    print(f"Table: Perfect duplicate group {group_id} | suggested keep = {group_table['suggested_keep'].iloc[0]}")
    display(group_table)

**After this cell:**

This is the first place to make manual decisions. For each duplicate group, pick one feature to keep and record why the others should be dropped later. Nothing is dropped in this notebook.

## 6. Missing Date Investigation

The missing rows are specific calendar events, not broad random missingness. Inspect the dates before deciding whether to drop rows, add a data-quality flag, or impute.

In [ ]:
missing_mask = raw[feature_cols].isna()
missing_row_counts = missing_mask.sum(axis=1)
missing_rows_df = raw.loc[missing_row_counts > 0, ["date", "instrument", "primary_signal"]].copy()
missing_rows_df["missing_feature_count"] = missing_row_counts.loc[missing_row_counts > 0].astype(int).to_numpy()
missing_rows_df["missing_feature_names"] = [
    ";".join(missing_mask.columns[row_values].tolist())
    for row_values in missing_mask.loc[missing_row_counts > 0].to_numpy()
]

# Dynamically find the window around the worst missing-value date
if len(missing_rows_df) > 0:
    worst_date = missing_rows_df.sort_values("missing_feature_count", ascending=False)["date"].iloc[0]
    context_start = worst_date - pd.Timedelta(days=4)
    context_end = worst_date + pd.Timedelta(days=4)
    context_df = raw.loc[raw["date"].between(context_start, context_end)].copy()
else:
    context_df = pd.DataFrame()

raw_context_cols = ["date", "instrument", "primary_signal", "open", "high", "low", "close", "volume", "open_interest"]
hmm_context_cols = [
    "date", "instrument", "primary_signal",
    "hmm_predicted_regime", "hmm_regime_confidence",
    "hmm_prob_stress", "hmm_prob_calm_positive", "hmm_prob_calm_negative", "hmm_prob_upside_breakout",
]
engineered_missing_context_cols = [
    "date", "instrument", "volume", "open_interest",
    "volume_log_change", "open_interest_log_change", "volume_to_open_interest",
]

print("Table: Rows with any missing feature values")
display(missing_rows_df)

if not context_df.empty:
    print(f"Table: Raw market context around worst missing date ({worst_date.date()})")
    display(context_df[[col for col in raw_context_cols if col in context_df.columns]])

    print("Table: HMM context around worst missing date")
    display(context_df[[col for col in hmm_context_cols if col in context_df.columns]])

    print("Table: Engineered volume/open-interest context around worst missing date")
    display(context_df[[col for col in engineered_missing_context_cols if col in context_df.columns]])

**After this cell:**

Missing rows are market holidays (all features NaN) and any HMM-gap dates. The context tables show the surrounding rows so you can confirm the cause before deciding to drop.

## 7. High-Correlation Deep Dive

High correlation is not automatically a drop signal. TA windows are often highly correlated because they describe related horizons. This section separates correlation pairs by type so you can inspect them more sensibly.

In [ ]:
numeric_df = raw[numeric_feature_cols]
spearman_corr = numeric_df.corr(method="spearman")
notna_matrix = numeric_df.notna().astype("int8")
pairwise_counts = notna_matrix.T.dot(notna_matrix)

def family_from_feature(feature: str) -> str:
    if feature.startswith("hmm_") or "_hmm_" in feature:
        return "hmm"
    if feature in ["open", "high", "low", "close", "volume", "open_interest"]:
        return "raw_market"
    if any(token in feature for token in ["ret_", "roc_", "mom_", "slope", "sma", "ema", "macd", "rsi", "stoch", "williams", "cci", "mfi", "uo", "zscore", "bb_", "donchian", "atr", "vol_"]):
        return "technical"
    if feature in ["volume_log_change", "open_interest_log_change", "volume_zscore_20d", "volume_zscore_63d", "open_interest_zscore_20d", "open_interest_zscore_63d", "volume_to_open_interest", "range_pct", "body_pct", "upper_wick_pct", "lower_wick_pct", "close_position_in_bar", "gap_open_pct", "vol_20d_for_interaction"]:
        return "engineered_market"
    if feature.startswith("signal_") or feature.startswith("days_since_signal"):
        return "engineered_signal"
    return "other"

def correlation_bucket(row: pd.Series) -> str:
    families = {row["feature_1_family"], row["feature_2_family"]}
    if "hmm" in families:
        return "HMM duplicates/redundancies"
    if families == {"technical"}:
        return "TA window variants"
    if families == {"raw_market"}:
        return "Raw price-level correlations"
    if "technical" in families and "engineered_market" in families:
        return "Engineered-vs-technical overlaps"
    return "Other high-correlation pairs"

high_corr_rows = []
for i, feature_1 in enumerate(numeric_feature_cols):
    for feature_2 in numeric_feature_cols[i + 1:]:
        corr_value = spearman_corr.loc[feature_1, feature_2]
        if pd.isna(corr_value):
            continue
        abs_corr = abs(corr_value)
        if abs_corr >= HIGH_CORR_THRESHOLD:
            high_corr_rows.append(
                {
                    "feature_1": feature_1,
                    "feature_2": feature_2,
                    "feature_1_group": feature_group_map.get(feature_1, "uncatalogued"),
                    "feature_2_group": feature_group_map.get(feature_2, "uncatalogued"),
                    "feature_1_family": family_from_feature(feature_1),
                    "feature_2_family": family_from_feature(feature_2),
                    "spearman_corr": corr_value,
                    "abs_spearman_corr": abs_corr,
                    "n_pairwise_non_missing": int(pairwise_counts.loc[feature_1, feature_2]),
                }
            )

high_corr_pairs = (
    pd.DataFrame(high_corr_rows)
    .sort_values(["abs_spearman_corr", "n_pairwise_non_missing", "feature_1", "feature_2"], ascending=[False, False, True, True])
    .reset_index(drop=True)
    if high_corr_rows
    else pd.DataFrame(columns=["feature_1", "feature_2", "feature_1_group", "feature_2_group", "feature_1_family", "feature_2_family", "spearman_corr", "abs_spearman_corr", "n_pairwise_non_missing"])
)

if not high_corr_pairs.empty:
    high_corr_pairs["correlation_bucket"] = high_corr_pairs.apply(correlation_bucket, axis=1)
    high_corr_pairs["manual_category"] = ""
    high_corr_pairs["manual_action"] = ""
    high_corr_pairs["manual_notes"] = ""
else:
    high_corr_pairs["correlation_bucket"] = []

high_corr_features = set(high_corr_pairs["feature_1"]).union(set(high_corr_pairs["feature_2"])) if not high_corr_pairs.empty else set()
max_abs_spearman_corr = pd.Series(0.0, index=feature_cols)
if not high_corr_pairs.empty:
    stacked = pd.concat(
        [
            high_corr_pairs[["feature_1", "abs_spearman_corr"]].rename(columns={"feature_1": "feature"}),
            high_corr_pairs[["feature_2", "abs_spearman_corr"]].rename(columns={"feature_2": "feature"}),
        ],
        ignore_index=True,
    )
    max_abs_spearman_corr = stacked.groupby("feature")["abs_spearman_corr"].max().reindex(feature_cols).fillna(0.0)

print(f"High-correlation threshold: absolute Spearman >= {HIGH_CORR_THRESHOLD}")
print(f"High-correlation pairs: {len(high_corr_pairs)}")
print(f"Features in at least one high-correlation pair: {len(high_corr_features)}")
if not high_corr_pairs.empty:
    print("Table: High-correlation pair counts by review bucket")
    display(high_corr_pairs.groupby("correlation_bucket").size().to_frame("pair_count").sort_values("pair_count", ascending=False))
    for bucket_name in high_corr_pairs["correlation_bucket"].drop_duplicates().tolist():
        bucket_table = high_corr_pairs.loc[high_corr_pairs["correlation_bucket"].eq(bucket_name)].copy()
        print(f"Table: High-correlation bucket = {bucket_name} | rows = {len(bucket_table)}")
        display(bucket_table)

**After this cell:**

Use the buckets differently. HMM redundancy can often be simplified aggressively. TA window variants need judgement, because nearby windows can be correlated but still represent different horizons.

## 8. Build manual feature review dataframe
`feature_review_df` is the master review surface. It includes automatic issue flags plus blank manual decision columns. This now also flags low-variance numeric features using `LOW_VARIANCE_THRESHOLD`.


In [ ]:
numeric_variance = raw[numeric_feature_cols].var(skipna=True)
constant_features = set(col for col in feature_cols if raw[col].nunique(dropna=True) <= 1)
low_variance_features = set(numeric_variance[numeric_variance <= LOW_VARIANCE_THRESHOLD].index)

def build_issue_flags(row: pd.Series) -> str:
    flags = []
    if row["is_non_numeric"]:
        flags.append("non_numeric")
    if row["missing_pct"] > 0:
        flags.append("missing_values")
    if row["is_constant"]:
        flags.append("constant")
    if row["is_low_variance"]:
        flags.append("low_variance")
    if row["is_duplicate"]:
        flags.append("duplicate")
    if row["high_corr_flag"]:
        flags.append("high_corr")
    return ";".join(flags)

feature_review_df = pd.DataFrame(
    {
        "feature": feature_cols,
        "feature_group": [feature_group_map.get(col, "uncatalogued") for col in feature_cols],
        "feature_family": [family_from_feature(col) for col in feature_cols],
        "dtype": [str(raw[col].dtype) for col in feature_cols],
        "missing_pct": [missing_pct[col] for col in feature_cols],
        "unique_values": [int(raw[col].nunique(dropna=True)) for col in feature_cols],
        "variance": [float(numeric_variance[col]) if col in numeric_variance.index and pd.notna(numeric_variance[col]) else pd.NA for col in feature_cols],
        "is_non_numeric": [col in non_numeric_feature_cols for col in feature_cols],
        "is_constant": [col in constant_features for col in feature_cols],
        "is_low_variance": [col in low_variance_features for col in feature_cols],
        "is_duplicate": [col in duplicate_features for col in feature_cols],
        "duplicate_of": [duplicate_of_map.get(col, pd.NA) for col in feature_cols],
        "duplicate_group_id": [
            duplicate_groups_df.loc[duplicate_groups_df["feature"].eq(col), "duplicate_group_id"].iloc[0]
            if not duplicate_groups_df.empty and duplicate_groups_df["feature"].eq(col).any()
            else pd.NA
            for col in feature_cols
        ],
        "max_abs_spearman_corr": [float(max_abs_spearman_corr.get(col, 0.0)) for col in feature_cols],
        "high_corr_flag": [col in high_corr_features for col in feature_cols],
    }
)
feature_review_df["auto_issue_flags"] = feature_review_df.apply(build_issue_flags, axis=1)
feature_review_df["manual_category"] = ""
feature_review_df["manual_action"] = ""
feature_review_df["manual_notes"] = ""

feature_review_df = feature_review_df.sort_values(
    ["auto_issue_flags", "feature_group", "feature"],
    ascending=[False, True, True],
).reset_index(drop=True)

print(f"Low-variance threshold: variance <= {LOW_VARIANCE_THRESHOLD}")
print(f"Low-variance numeric features for review instrument: {len(low_variance_features)}")
if low_variance_features:
    print("Table: Low-variance features")
    display(
        feature_review_df.loc[feature_review_df["is_low_variance"], [
            "feature", "feature_group", "feature_family", "dtype", "variance", "unique_values", "missing_pct"
        ]].sort_values(["variance", "feature"])
    )

print("Table: Manual feature review dataframe")
display(feature_review_df)

print("Table: Auto issue flag counts")
display(
    feature_review_df.assign(has_issue=feature_review_df["auto_issue_flags"].ne(""))
    .groupby(["feature_group", "has_issue"])
    .size()
    .unstack(fill_value=0)
)


**After this cell:**

This is the table to use while categorising features manually. It does not enforce decisions; it only makes the problems visible.

## 9. Feature group review tables

These grouped views make manual inspection easier by showing related feature families together.

In [ ]:
for group_name in sorted(feature_review_df["feature_group"].unique()):
    group_table = feature_review_df.loc[feature_review_df["feature_group"].eq(group_name)].copy()
    print(f"Table: Feature review group = {group_name} | rows = {len(group_table)}")
    display(group_table)

**After this cell:**

Reviewing by feature group is less overwhelming than scanning all features at once. Start with HMM groups and duplicate groups before worrying about TA window correlations.

## 10. Save cleaned model datasets
Applies the cleaning decisions that are obvious from the review tables: exact HMM aliases, constant HMM columns, duplicate missing flags, non-numeric labels, simple TA overlaps, and low-variance numeric features. The same hard-drop rules are applied to each energy instrument in `ENERGY_INSTRUMENTS`; low-variance columns are then detected and removed separately per instrument. Rows with any remaining missing feature values are dropped per instrument. Output is saved to `data/features/clean_feature_set/{instrument}_clean_feature_set.csv`.


In [ ]:
hmm_constant_drop_features = ["hmm_prob_chop", "hmm_market_chop"]

hmm_missing_flag_drop_features = [
    "hmm_volatility_regime_missing",
    "hmm_return_regime_missing",
    "hmm_market_state_missing",
]

hmm_probability_alias_drop_features = [
    "hmm_prob_extreme_vol", "hmm_prob_downside", "hmm_regime_prob_0",
    "hmm_prob_normal_vol", "hmm_prob_positive", "hmm_regime_prob_1",
    "hmm_prob_low_vol", "hmm_prob_weak", "hmm_regime_prob_2",
    "hmm_prob_high_vol", "hmm_prob_strong_upside", "hmm_regime_prob_3",
    "hmm_prob_not_stress",
]

hmm_one_hot_alias_drop_features = [
    "hmm_regime_0", "hmm_regime_1", "hmm_regime_2", "hmm_regime_3",
    "hmm_volatility_extreme_vol", "hmm_volatility_normal_vol",
    "hmm_volatility_low_vol", "hmm_volatility_high_vol",
    "hmm_return_downside", "hmm_return_positive",
    "hmm_return_weak", "hmm_return_strong_upside",
]

non_numeric_hmm_label_drop_features = [
    "hmm_volatility_regime", "hmm_return_regime", "hmm_market_state",
]

hmm_category_code_drop_features = [
    "hmm_volatility_regime_code", "hmm_return_regime_code", "hmm_market_state_code",
]

simple_ta_alias_drop_features = [
    "roc_5d", "roc_20d", "roc_63d", "bb_position", "overnight_ret", "intraday_ret",
]

return_ladder_drop_features = ["log_ret", "ret_2d", "ret_3d", "ret_10d"]

requested_hard_drop_features = (
    hmm_constant_drop_features
    + hmm_missing_flag_drop_features
    + hmm_probability_alias_drop_features
    + hmm_one_hot_alias_drop_features
    + non_numeric_hmm_label_drop_features
    + hmm_category_code_drop_features
    + simple_ta_alias_drop_features
    + return_ladder_drop_features
)
hard_drop_features = [f for f in requested_hard_drop_features if f in full_matrix.columns]

id_cols = ["date", "instrument"]
signal_col = "primary_signal"
CLEAN_FEATURE_SET_DIR.mkdir(parents=True, exist_ok=True)

clean_feature_sets = {}
clean_dataset_summaries = []
low_variance_drop_features_by_instrument = {}

for instrument in ENERGY_INSTRUMENTS:
    instrument_raw = full_matrix.loc[full_matrix["instrument"].eq(instrument)].reset_index(drop=True)
    assert len(instrument_raw) > 0, f"No rows found for {instrument}."
    assert instrument_raw["date"].duplicated().sum() == 0, f"Duplicate dates found for {instrument}."

    # 1) Apply the manual/global hard drops.
    instrument_clean = instrument_raw.drop(columns=hard_drop_features).copy()

    # 2) First pass: remove low-variance numeric features for this instrument.
    # This pass is before the missing-row drop, so it catches globally constant/near-constant columns.
    instrument_model_feature_cols = [col for col in instrument_clean.columns if col not in id_cols + [signal_col]]
    instrument_numeric_feature_cols = instrument_clean[instrument_model_feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    instrument_variance = instrument_clean[instrument_numeric_feature_cols].var(skipna=True)
    low_variance_drop_features = instrument_variance[instrument_variance <= LOW_VARIANCE_THRESHOLD].index.tolist()

    if low_variance_drop_features:
        instrument_clean = instrument_clean.drop(columns=low_variance_drop_features).copy()

    # 3) Drop rows that still have missing feature values.
    instrument_model_feature_cols = [col for col in instrument_clean.columns if col not in id_cols + [signal_col]]
    missing_after_drop = instrument_clean[instrument_model_feature_cols].isna().any(axis=1)
    instrument_clean = instrument_clean.loc[~missing_after_drop].reset_index(drop=True)

    # 4) Second pass: recompute variance after row removal.
    # Some features can become low-variance only after missing rows are removed.
    instrument_model_feature_cols = [col for col in instrument_clean.columns if col not in id_cols + [signal_col]]
    instrument_numeric_feature_cols = instrument_clean[instrument_model_feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    instrument_variance_after_rows = instrument_clean[instrument_numeric_feature_cols].var(skipna=True)
    post_missing_low_variance_drop_features = instrument_variance_after_rows[
        instrument_variance_after_rows <= LOW_VARIANCE_THRESHOLD
    ].index.tolist()

    if post_missing_low_variance_drop_features:
        instrument_clean = instrument_clean.drop(columns=post_missing_low_variance_drop_features).copy()

    # Store the complete low-variance drop list for validation/reporting.
    low_variance_drop_features_by_instrument[instrument] = sorted(
        set(low_variance_drop_features + post_missing_low_variance_drop_features)
    )

    output_path = CLEAN_FEATURE_SET_PATHS[instrument]
    instrument_clean.to_csv(output_path, index=False)
    clean_feature_sets[instrument] = instrument_clean

    clean_dataset_summaries.append(
        {
            "instrument": instrument,
            "original_rows": len(instrument_raw),
            "clean_rows": len(instrument_clean),
            "rows_removed": len(instrument_raw) - len(instrument_clean),
            "original_columns": instrument_raw.shape[1],
            "clean_columns": instrument_clean.shape[1],
            "hard_drop_features_removed": len(hard_drop_features),
            "low_variance_features_removed": len(low_variance_drop_features),
            "total_features_removed": len(hard_drop_features) + len(low_variance_drop_features),
            "remaining_feature_columns": len(instrument_model_feature_cols),
            "low_variance_removed_features": "; ".join(low_variance_drop_features),
            "output_path": str(output_path),
        }
    )

# Keep the review instrument variables available for the validation cells below.
clean = clean_feature_sets[INSTRUMENT]
model_feature_cols = [col for col in clean.columns if col not in id_cols + [signal_col]]
CLEAN_FEATURE_SET_PATH = CLEAN_FEATURE_SET_PATHS[INSTRUMENT]

hard_drop_summary = pd.DataFrame(
    {
        "drop_group": (
            ["constant_hmm"] * len(hmm_constant_drop_features)
            + ["duplicate_missing_flag"] * len(hmm_missing_flag_drop_features)
            + ["hmm_probability_alias"] * len(hmm_probability_alias_drop_features)
            + ["hmm_one_hot_alias"] * len(hmm_one_hot_alias_drop_features)
            + ["non_numeric_hmm_label"] * len(non_numeric_hmm_label_drop_features)
            + ["hmm_category_code"] * len(hmm_category_code_drop_features)
            + ["simple_ta_alias"] * len(simple_ta_alias_drop_features)
            + ["return_ladder_trim"] * len(return_ladder_drop_features)
        ),
        "feature": requested_hard_drop_features,
    }
).loc[lambda df: df["feature"].isin(hard_drop_features)].reset_index(drop=True)

low_variance_drop_summary = (
    pd.DataFrame(
        [
            {"instrument": instrument, "feature": feature}
            for instrument, features in low_variance_drop_features_by_instrument.items()
            for feature in features
        ],
        columns=["instrument", "feature"],
    )
)

clean_dataset_summary = pd.DataFrame(clean_dataset_summaries)
model_dataset_summary = clean_dataset_summary.loc[clean_dataset_summary["instrument"].eq(INSTRUMENT)].T.rename(columns={0: "value"})

print("Table: Hard-drop feature list")
display(hard_drop_summary)

print(f"Table: Low-variance feature list by instrument | threshold = {LOW_VARIANCE_THRESHOLD}")
display(low_variance_drop_summary)

print("Table: Clean dataset summary for all energy instruments")
display(clean_dataset_summary)

print(f"Table: Clean dataset sample for {INSTRUMENT}")
display(clean.head())

print("Saved clean feature sets:")
for instrument, path in CLEAN_FEATURE_SET_PATHS.items():
    print(f"- {instrument}: {path}")


**After this cell:** Clean datasets are saved in `data/features/clean_feature_set/`. Duplicate/alias features and low-variance numeric features are removed, and rows with missing values are dropped separately for each energy instrument. High-correlation TA families that are not simple aliases remain for later CPCV-safe selection.


## 11. Validation

Confirms every energy clean dataset was saved correctly: no duplicate dates, no remaining missing feature values, no hard-dropped columns, and `primary_signal` is retained.

In [ ]:
assert INSTRUMENT in full_matrix["instrument"].values, f"{INSTRUMENT} not found in feature matrix."
assert len(raw) > 0, "No rows after filtering to instrument."
assert duplicate_dates == 0, "Duplicate dates found for review instrument."
assert len(feature_review_df) == len(feature_cols), "feature_review_df should have one row per feature column."
assert set(feature_review_df["feature"]) == set(feature_cols), "feature_review_df feature set does not match feature columns."
assert "feature_review_df" in globals(), "feature_review_df was not created."
assert "high_corr_pairs" in globals(), "high_corr_pairs was not created."
assert "duplicate_feature_pairs" in globals(), "duplicate_feature_pairs was not created."
assert "duplicate_groups_df" in globals(), "duplicate_groups_df was not created."
assert "clean_feature_sets" in globals(), "clean_feature_sets was not created."
assert "low_variance_drop_features_by_instrument" in globals(), "low_variance_drop_features_by_instrument was not created."

validation_rows = []
for instrument in ENERGY_INSTRUMENTS:
    path = CLEAN_FEATURE_SET_PATHS[instrument]
    assert path.exists(), f"Output file was not saved: {path}"

    saved = pd.read_csv(path, parse_dates=["date"])
    saved_feature_cols = [col for col in saved.columns if col not in id_cols + [signal_col]]
    saved_numeric_feature_cols = saved[saved_feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    saved_variance = saved[saved_numeric_feature_cols].var(skipna=True)
    saved_low_variance = saved_variance[saved_variance <= LOW_VARIANCE_THRESHOLD].sort_values()

    assert len(saved) > 0, f"Clean dataset is empty for {instrument}."
    assert saved["instrument"].eq(instrument).all(), f"Unexpected instrument values in {path}."
    assert saved["date"].duplicated().sum() == 0, f"Duplicate dates found in {path}."
    assert not set(hard_drop_features).intersection(saved.columns), f"Hard-drop features still present in {path}."
    assert not set(low_variance_drop_features_by_instrument[instrument]).intersection(saved.columns), f"Low-variance features still present in {path}."
    assert saved_low_variance.empty, (
        f"Saved dataset still has low-variance numeric features: {path}. "
        f"Features: {saved_low_variance.index.tolist()}"
    )
    assert saved[saved_feature_cols].isna().sum().sum() == 0, f"Clean dataset still has missing feature values: {path}."
    assert signal_col in saved.columns, f"primary_signal column is missing from {path}."

    validation_rows.append(
        {
            "instrument": instrument,
            "rows": len(saved),
            "columns": saved.shape[1],
            "feature_columns": len(saved_feature_cols),
            "min_date": saved["date"].min(),
            "max_date": saved["date"].max(),
            "primary_signal_values": sorted(saved[signal_col].dropna().unique().tolist()),
            "path": str(path),
        }
    )

validation_summary = pd.DataFrame(validation_rows)
print("Validation passed for all clean energy feature sets.")
display(validation_summary)


**After this cell:** All four clean feature sets are ready for the modelling phase and available from `clean_feature_sets` in memory or from the CSV files under `data/features/clean_feature_set/`.